# Deploying LLM Endpoints as APIs in Dataiku

## Learning Objectives

By completing this notebook, you will:
1. Deploy LLM applications as REST API endpoints in Dataiku
2. Configure authentication and access control for APIs
3. Implement request validation and error handling
4. Design versioned API endpoints for backward compatibility
5. Test and validate deployed endpoints

## Prerequisites

- Module 3 completed (Custom LLM applications)
- Understanding of REST API concepts
- Familiarity with HTTP methods and status codes
- Basic knowledge of API authentication

## Estimated Time: 60 minutes

---

## Why Deploy as APIs?

**From notebook to service**: Production LLM applications need to be accessible to other systems.

API deployment enables:
- **Integration** - Connect to web apps, mobile apps, other services
- **Scalability** - Handle concurrent requests
- **Versioning** - Support multiple API versions simultaneously
- **Security** - Implement authentication and rate limiting
- **Monitoring** - Track usage, performance, errors

### Key Insight

**APIs are the bridge** between your LLM application and the real world.

### Dataiku API Endpoints

Dataiku provides:
1. **Python function endpoints** - Deploy Python functions as APIs
2. **Model endpoints** - Serve ML/LLM models
3. **Dataset endpoints** - Query datasets via API
4. **SQL query endpoints** - Execute SQL queries
5. **R function endpoints** - Deploy R functions

We'll focus on Python function endpoints for LLM applications.

## Setup

Import libraries for API development and testing.

In [ ]:
# Purpose: Setup API development environment
# Key Concept: API endpoints require structured request/response handling

import dataiku
from dataiku.llm import LLM
import json
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict
from enum import Enum
import time
from datetime import datetime
import hashlib
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('APIEndpoint')

print("✓ API development environment ready")

## API Contract Design

Good APIs start with clear contracts specifying:
- Request format
- Response format
- Error responses
- Status codes
- Authentication

In [ ]:
# Purpose: Define API contracts
# Key Concept: Clear contracts prevent integration issues

class APIVersion(Enum):
    """API version enumeration."""
    V1 = "v1"
    V2 = "v2"

class HTTPStatus(Enum):
    """HTTP status codes."""
    OK = 200
    CREATED = 201
    BAD_REQUEST = 400
    UNAUTHORIZED = 401
    NOT_FOUND = 404
    INTERNAL_ERROR = 500
    SERVICE_UNAVAILABLE = 503

@dataclass
class APIRequest:
    """
    Standard API request format.
    
    All API requests should follow this structure.
    """
    # Request metadata
    request_id: str
    api_version: str
    
    # Request data
    endpoint: str
    data: Dict[str, Any]
    
    # Optional fields
    timestamp: Optional[str] = None
    client_id: Optional[str] = None

@dataclass
class APIResponse:
    """
    Standard API response format.
    
    All API responses should follow this structure.
    """
    # Response metadata
    request_id: str
    status_code: int
    status: str  # 'success' or 'error'
    
    # Response data
    data: Optional[Dict[str, Any]] = None
    error: Optional[str] = None
    
    # Performance metadata
    processing_time_ms: Optional[float] = None
    timestamp: Optional[str] = None

@dataclass
class AnalyzeRequest:
    """Request for commodity analysis endpoint."""
    report_text: str
    commodity: Optional[str] = None
    include_price_target: bool = False

@dataclass
class AnalyzeResponse:
    """Response from commodity analysis endpoint."""
    commodity: str
    sentiment: str
    confidence: str
    key_drivers: List[str]
    risk_level: str
    summary: str
    price_target: Optional[float] = None

print("✓ API contracts defined")

# Example request/response
example_request = {
    "request_id": "req_123456",
    "api_version": "v1",
    "endpoint": "/analyze",
    "data": {
        "report_text": "Oil prices surged 5% on supply concerns.",
        "commodity": "Crude Oil",
        "include_price_target": True
    }
}

example_response = {
    "request_id": "req_123456",
    "status_code": 200,
    "status": "success",
    "data": {
        "commodity": "Crude Oil",
        "sentiment": "bullish",
        "confidence": "high",
        "key_drivers": ["supply concerns", "geopolitical tensions"],
        "risk_level": "medium",
        "summary": "Strong bullish signal driven by supply constraints."
    },
    "processing_time_ms": 1234.5,
    "timestamp": "2024-02-03T10:00:00Z"
}

print("\nExample Request:")
print(json.dumps(example_request, indent=2))
print("\nExample Response:")
print(json.dumps(example_response, indent=2))

## Request Validation

Validate all incoming requests before processing.

In [ ]:
# Purpose: Implement request validation
# Key Concept: Validation prevents invalid data from reaching your LLM

class ValidationError(Exception):
    """Raised when request validation fails."""
    pass

class RequestValidator:
    """
    Validate API requests.
    
    Checks:
    - Required fields present
    - Data types correct
    - Values within acceptable ranges
    - Format specifications met
    """
    
    @staticmethod
    def validate_analyze_request(data: Dict) -> AnalyzeRequest:
        """
        Validate and parse analysis request.
        
        Parameters
        ----------
        data : dict
            Raw request data
        
        Returns
        -------
        AnalyzeRequest
            Validated request object
        
        Raises
        ------
        ValidationError
            If validation fails
        """
        # Check required fields
        if 'report_text' not in data:
            raise ValidationError("Missing required field: report_text")
        
        report_text = data['report_text']
        
        # Validate report_text
        if not isinstance(report_text, str):
            raise ValidationError("report_text must be a string")
        
        if len(report_text.strip()) == 0:
            raise ValidationError("report_text cannot be empty")
        
        if len(report_text) > 5000:
            raise ValidationError("report_text exceeds maximum length of 5000 characters")
        
        # Validate optional commodity
        commodity = data.get('commodity')
        if commodity is not None:
            if not isinstance(commodity, str):
                raise ValidationError("commodity must be a string")
            
            valid_commodities = [
                'Crude Oil', 'Natural Gas', 'Gold', 'Silver', 'Copper',
                'Corn', 'Wheat', 'Soybeans', 'Coffee', 'Cotton'
            ]
            if commodity not in valid_commodities:
                raise ValidationError(
                    f"Invalid commodity. Must be one of: {', '.join(valid_commodities)}"
                )
        
        # Validate optional include_price_target
        include_price_target = data.get('include_price_target', False)
        if not isinstance(include_price_target, bool):
            raise ValidationError("include_price_target must be a boolean")
        
        return AnalyzeRequest(
            report_text=report_text,
            commodity=commodity,
            include_price_target=include_price_target
        )
    
    @staticmethod
    def validate_request_metadata(data: Dict) -> Dict:
        """
        Validate request metadata (request_id, api_version, etc.).
        
        Returns
        -------
        dict
            Validated metadata
        """
        metadata = {}
        
        # request_id
        if 'request_id' in data:
            if not isinstance(data['request_id'], str):
                raise ValidationError("request_id must be a string")
            metadata['request_id'] = data['request_id']
        else:
            # Generate if not provided
            metadata['request_id'] = f"req_{int(time.time() * 1000)}"
        
        # api_version
        api_version = data.get('api_version', 'v1')
        if api_version not in ['v1', 'v2']:
            raise ValidationError(f"Unsupported API version: {api_version}")
        metadata['api_version'] = api_version
        
        return metadata

# Test validation
validator = RequestValidator()

print("Testing request validation...\n")

# Valid request
valid_data = {
    'report_text': 'Oil prices rose 3% today.',
    'commodity': 'Crude Oil',
    'include_price_target': True
}

try:
    request = validator.validate_analyze_request(valid_data)
    print("✓ Valid request accepted")
    print(f"  Commodity: {request.commodity}")
    print(f"  Include price target: {request.include_price_target}")
except ValidationError as e:
    print(f"✗ Validation failed: {e}")

print()

# Invalid requests
invalid_requests = [
    ({}, "Missing report_text"),
    ({'report_text': ''}, "Empty report_text"),
    ({'report_text': 'Test', 'commodity': 'Bitcoin'}, "Invalid commodity"),
    ({'report_text': 'Test', 'include_price_target': 'yes'}, "Wrong type for boolean")
]

for invalid_data, expected_error in invalid_requests:
    try:
        validator.validate_analyze_request(invalid_data)
        print(f"✗ Should have rejected: {expected_error}")
    except ValidationError as e:
        print(f"✓ Correctly rejected: {expected_error}")

print("\n✓ Validation system tested")

## Exercise 1: Build a Complete API Endpoint

**Task**: Create a production-ready API endpoint with validation, error handling, and monitoring.

Your endpoint should:
- Validate all inputs
- Handle errors gracefully
- Return proper HTTP status codes
- Log all requests and responses
- Track performance metrics

In [ ]:
# YOUR CODE HERE

class CommodityAnalysisEndpoint:
    """
    Production-ready API endpoint for commodity analysis.
    
    This would be deployed as a Dataiku Python function endpoint.
    """
    
    def __init__(self):
        # Initialize once when endpoint starts
        self.llm = LLM('claude-sonnet')
        self.validator = RequestValidator()
        
        # Simple in-memory cache for demo
        self.cache = {}
        
        # Metrics
        self.request_count = 0
        self.error_count = 0
    
    def handle(self, raw_request: Dict) -> Dict:
        """
        Main endpoint handler.
        
        This is the function Dataiku would call for each API request.
        
        Parameters
        ----------
        raw_request : dict
            Raw request data from API call
        
        Returns
        -------
        dict
            API response
        """
        start_time = time.time()
        self.request_count += 1
        
        try:
            # Step 1: Validate metadata
            # YOUR CODE HERE
            metadata = self.validator.validate_request_metadata(raw_request)
            request_id = metadata['request_id']
            
            logger.info(f"Processing request {request_id}")
            
            # Step 2: Validate request data
            if 'data' not in raw_request:
                return self._error_response(
                    request_id,
                    HTTPStatus.BAD_REQUEST.value,
                    "Missing 'data' field",
                    start_time
                )
            
            request_data = self.validator.validate_analyze_request(raw_request['data'])
            
            # Step 3: Process request
            result = self._process_analysis(request_data)
            
            # Step 4: Build success response
            processing_time = (time.time() - start_time) * 1000
            
            response = APIResponse(
                request_id=request_id,
                status_code=HTTPStatus.OK.value,
                status='success',
                data=asdict(result),
                processing_time_ms=processing_time,
                timestamp=datetime.now().isoformat()
            )
            
            logger.info(f"Request {request_id} completed successfully ({processing_time:.0f}ms)")
            return asdict(response)
            
        except ValidationError as e:
            self.error_count += 1
            logger.warning(f"Validation error: {e}")
            return self._error_response(
                raw_request.get('request_id', 'unknown'),
                HTTPStatus.BAD_REQUEST.value,
                str(e),
                start_time
            )
        
        except Exception as e:
            self.error_count += 1
            logger.error(f"Internal error: {e}", exc_info=True)
            return self._error_response(
                raw_request.get('request_id', 'unknown'),
                HTTPStatus.INTERNAL_ERROR.value,
                "Internal server error",
                start_time
            )
    
    def _process_analysis(self, request: AnalyzeRequest) -> AnalyzeResponse:
        """
        Process the analysis request using LLM.
        """
        # Build prompt
        prompt = f"""You are an expert commodity market analyst.

Analyze this report and provide structured insights.

Report: {request.report_text}
{f'Commodity: {request.commodity}' if request.commodity else ''}

Provide analysis in JSON format:
{{
  "commodity": "name",
  "sentiment": "bullish"|"bearish"|"neutral",
  "confidence": "high"|"medium"|"low",
  "key_drivers": ["list", "of", "factors"],
  "risk_level": "low"|"medium"|"high",
  "summary": "brief analysis"{',\n  "price_target": numeric_value' if request.include_price_target else ''}
}}

JSON:"""
        
        # Call LLM
        response = self.llm.complete(prompt, max_tokens=400, temperature=0.3)
        
        # Parse response
        data = json.loads(response.text)
        
        return AnalyzeResponse(
            commodity=data['commodity'],
            sentiment=data['sentiment'],
            confidence=data['confidence'],
            key_drivers=data['key_drivers'],
            risk_level=data['risk_level'],
            summary=data['summary'],
            price_target=data.get('price_target')
        )
    
    def _error_response(self, request_id: str, status_code: int, 
                       error_message: str, start_time: float) -> Dict:
        """Build standardized error response."""
        processing_time = (time.time() - start_time) * 1000
        
        response = APIResponse(
            request_id=request_id,
            status_code=status_code,
            status='error',
            error=error_message,
            processing_time_ms=processing_time,
            timestamp=datetime.now().isoformat()
        )
        
        return asdict(response)
    
    def get_metrics(self) -> Dict:
        """Get endpoint metrics."""
        return {
            'total_requests': self.request_count,
            'total_errors': self.error_count,
            'error_rate': self.error_count / self.request_count if self.request_count > 0 else 0
        }

# Test the endpoint
endpoint = CommodityAnalysisEndpoint()

test_requests = [
    # Valid request
    {
        'request_id': 'test_001',
        'api_version': 'v1',
        'data': {
            'report_text': 'Crude oil surged 4.5% on OPEC+ production cuts.',
            'commodity': 'Crude Oil',
            'include_price_target': False
        }
    },
    # Invalid request (missing data)
    {
        'request_id': 'test_002',
        'api_version': 'v1'
    },
    # Valid request without commodity
    {
        'request_id': 'test_003',
        'api_version': 'v1',
        'data': {
            'report_text': 'Gold prices steady near record highs.'
        }
    }
]

print("=== Testing API Endpoint ===")
print()

for request in test_requests:
    response = endpoint.handle(request)
    
    print(f"Request ID: {response['request_id']}")
    print(f"Status: {response['status']} (HTTP {response['status_code']})")
    print(f"Processing time: {response['processing_time_ms']:.0f}ms")
    
    if response['status'] == 'success':
        data = response['data']
        print(f"  Commodity: {data['commodity']}")
        print(f"  Sentiment: {data['sentiment']} ({data['confidence']} confidence)")
    else:
        print(f"  Error: {response['error']}")
    print()

print("Endpoint Metrics:")
for key, value in endpoint.get_metrics().items():
    print(f"  {key}: {value}")

# Auto-graded checks
valid_response = endpoint.handle(test_requests[0])
assert valid_response['status'] == 'success', "Valid request should succeed"
assert valid_response['status_code'] == 200, "Success should return 200"

invalid_response = endpoint.handle(test_requests[1])
assert invalid_response['status'] == 'error', "Invalid request should return error"
assert invalid_response['status_code'] == 400, "Validation error should return 400"

print("\n✓ Exercise 1 passed!")

## API Versioning

Support multiple API versions for backward compatibility.

In [ ]:
# Purpose: Implement API versioning
# Key Concept: Versioning allows evolution without breaking existing clients

class VersionedEndpoint:
    """
    API endpoint with version support.
    
    Different versions can have different:
    - Response formats
    - Processing logic
    - Validation rules
    """
    
    def __init__(self):
        self.llm = LLM('claude-sonnet')
        self.validator = RequestValidator()
    
    def handle(self, raw_request: Dict) -> Dict:
        """Route to appropriate version handler."""
        api_version = raw_request.get('api_version', 'v1')
        
        if api_version == 'v1':
            return self._handle_v1(raw_request)
        elif api_version == 'v2':
            return self._handle_v2(raw_request)
        else:
            return {
                'status': 'error',
                'status_code': 400,
                'error': f"Unsupported API version: {api_version}"
            }
    
    def _handle_v1(self, request: Dict) -> Dict:
        """
        V1 endpoint - original format.
        
        Returns basic analysis without price target.
        """
        # V1 processing logic
        return {
            'api_version': 'v1',
            'status': 'success',
            'data': {
                'sentiment': 'bullish',
                'summary': 'V1 format response'
            }
        }
    
    def _handle_v2(self, request: Dict) -> Dict:
        """
        V2 endpoint - enhanced format.
        
        Returns detailed analysis with additional fields:
        - Price target
        - Risk assessment
        - Confidence scores
        """
        # V2 processing logic with enhanced features
        return {
            'api_version': 'v2',
            'status': 'success',
            'data': {
                'sentiment': 'bullish',
                'summary': 'V2 format response',
                'price_target': 85.50,
                'risk_score': 0.65,
                'confidence_scores': {
                    'sentiment': 0.92,
                    'price_target': 0.78
                }
            }
        }

# Test versioning
versioned = VersionedEndpoint()

print("=== Testing API Versioning ===")
print()

v1_request = {'api_version': 'v1', 'data': {'report_text': 'Test'}}
v1_response = versioned.handle(v1_request)
print("V1 Response:")
print(json.dumps(v1_response, indent=2))
print()

v2_request = {'api_version': 'v2', 'data': {'report_text': 'Test'}}
v2_response = versioned.handle(v2_request)
print("V2 Response (enhanced):")
print(json.dumps(v2_response, indent=2))
print()

print("✓ API versioning demonstrated")

## Exercise 2: API Testing Suite

**Task**: Create a comprehensive test suite for your API endpoint.

Your test suite should cover:
- Happy path scenarios
- Invalid inputs
- Edge cases
- Performance requirements
- Error handling

In [ ]:
# YOUR CODE HERE

class APITestSuite:
    """
    Comprehensive test suite for API endpoints.
    """
    
    def __init__(self, endpoint):
        self.endpoint = endpoint
        self.tests_passed = 0
        self.tests_failed = 0
    
    def test_valid_request(self):
        """Test that valid requests succeed."""
        request = {
            'request_id': 'test_valid',
            'api_version': 'v1',
            'data': {
                'report_text': 'Oil prices rose 3% on supply concerns.',
                'commodity': 'Crude Oil'
            }
        }
        
        response = self.endpoint.handle(request)
        
        assert response['status'] == 'success', "Valid request should succeed"
        assert response['status_code'] == 200, "Should return 200"
        assert 'data' in response, "Should include data"
        
        self.tests_passed += 1
        print("✓ test_valid_request passed")
    
    def test_missing_data(self):
        """Test that requests without data field are rejected."""
        request = {
            'request_id': 'test_missing_data',
            'api_version': 'v1'
        }
        
        response = self.endpoint.handle(request)
        
        assert response['status'] == 'error', "Should return error"
        assert response['status_code'] == 400, "Should return 400"
        
        self.tests_passed += 1
        print("✓ test_missing_data passed")
    
    # YOUR CODE HERE - Add more test methods:
    # - test_empty_report_text
    # - test_invalid_commodity
    # - test_response_time
    # - test_error_format
    
    def test_empty_report_text(self):
        """Test that empty report text is rejected."""
        request = {
            'request_id': 'test_empty',
            'api_version': 'v1',
            'data': {'report_text': ''}
        }
        
        response = self.endpoint.handle(request)
        assert response['status'] == 'error'
        assert response['status_code'] == 400
        
        self.tests_passed += 1
        print("✓ test_empty_report_text passed")
    
    def test_response_time(self):
        """Test that responses meet latency SLA."""
        request = {
            'request_id': 'test_latency',
            'api_version': 'v1',
            'data': {
                'report_text': 'Quick test for latency.'
            }
        }
        
        response = self.endpoint.handle(request)
        
        # SLA: 5 seconds max
        assert response['processing_time_ms'] < 5000, "Should respond within 5 seconds"
        
        self.tests_passed += 1
        print("✓ test_response_time passed")
    
    def run_all(self):
        """Run all tests."""
        print("=== Running API Test Suite ===")
        print()
        
        tests = [
            self.test_valid_request,
            self.test_missing_data,
            self.test_empty_report_text,
            self.test_response_time
        ]
        
        for test in tests:
            try:
                test()
            except AssertionError as e:
                self.tests_failed += 1
                print(f"✗ {test.__name__} failed: {e}")
            except Exception as e:
                self.tests_failed += 1
                print(f"✗ {test.__name__} error: {e}")
        
        print()
        print(f"Tests passed: {self.tests_passed}")
        print(f"Tests failed: {self.tests_failed}")
        print(f"Success rate: {self.tests_passed / (self.tests_passed + self.tests_failed):.1%}")

# Run the test suite
test_suite = APITestSuite(endpoint)
test_suite.run_all()

# Auto-graded checks
assert test_suite.tests_passed >= 4, "Should pass at least 4 tests"
assert test_suite.tests_failed == 0, "All tests should pass"

print("\n✓ Exercise 2 passed!")

## Summary

Congratulations! You've learned to deploy LLM applications as production APIs. Key takeaways:

1. **Clear contracts** - Define request/response formats upfront
2. **Validation** - Reject invalid inputs early
3. **Error handling** - Return proper HTTP status codes and messages
4. **Versioning** - Support multiple versions for compatibility
5. **Testing** - Comprehensive test coverage before deployment

## Deployment Checklist

Before deploying to production:

- [ ] API contract documented
- [ ] Request validation implemented
- [ ] Error handling covers all cases
- [ ] Proper HTTP status codes returned
- [ ] Test suite passes 100%
- [ ] Performance meets SLA
- [ ] Authentication configured
- [ ] Rate limiting enabled
- [ ] Logging comprehensive
- [ ] Monitoring dashboards ready

## Dataiku API Deployment Steps

1. **Create Python function endpoint**:
   - Navigate to API Designer
   - Create new endpoint
   - Select "Python function"

2. **Deploy code**:
   - Copy your endpoint class
   - Define handle() function
   - Test in API Designer

3. **Configure settings**:
   - Set authentication method
   - Configure rate limits
   - Enable CORS if needed

4. **Deploy to API service**:
   - Package endpoint
   - Deploy to infrastructure
   - Get production URL

5. **Monitor and iterate**:
   - Track usage metrics
   - Review error logs
   - Optimize performance

## Next Steps

- **Notebook 02**: Monitor API performance, costs, and quality
- Set up alerting for errors and SLA violations
- Create usage dashboards
- Implement A/B testing for prompt improvements